# Credit Train — Modelling

**Upstream:** `03_credit_feature_selection.ipynb`
**Features:** Standard tier (25) from `selected_features_standard.json`
**Target:** `Default` (1 = Charged Off, 0 = Fully Paid)

---

### Purpose

Train and compare baseline classifiers on the selected feature set, choose an operating threshold under class imbalance, and produce test-set scores for the champion model.

### Objectives

1. Leakage-safe train / validation split
2. Baseline models: Logistic Regression, Random Forest, Gradient Boosting
3. Discrimination metrics: ROC-AUC, PR-AUC, precision, recall, F1
4. Threshold tuning for business-relevant operating points
5. Calibration check and segment performance
6. Refit champion on full train and score `credit_test.csv`
7. Persist model, metrics, and predictions


## 0. Setup

Import modelling libraries, fix random seeds, and point to the Standard feature list and output directory.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)

ROOT = Path("../..")
FEAT_DIR = Path("features")
OUT_DIR = Path("model_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FEAT = FEAT_DIR / "credit_train_features.csv"
SELECT = FEAT_DIR / "selected_features_standard.json"
RAW_TEST = ROOT / "Datasets" / "credit_test.csv"
RAW_TRAIN = ROOT / "Datasets" / "credit_train.csv"

RANDOM_STATE = 42
print("Ready.")


## 1. Load selected features

Read the engineered training matrix and restrict columns to the **Standard (25)** set from feature selection.


In [ ]:
feat = pd.read_csv(TRAIN_FEAT)
with open(SELECT) as f:
    sel = json.load(f)

feature_cols = sel["features"]
target_col = sel.get("target", "Default")
feature_cols = [c for c in feature_cols if c in feat.columns]

X = feat[feature_cols].copy()
y = feat[target_col].copy()
ids = feat[["Loan ID", "Customer ID"]].copy() if "Loan ID" in feat.columns else None

print(f"X: {X.shape}  |  default rate: {y.mean()*100:.2f}%")
print(f"Features ({len(feature_cols)}):")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {c}")


## 2. Train / validation split

Create a stratified 80 / 20 hold-out so class balance is preserved in both folds.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  default={y_train.mean()*100:.2f}%")
print(f"Val:   {X_val.shape}  default={y_val.mean()*100:.2f}%")


## 3. Preprocessing + models

- **Numeric pipeline:** median impute → standard scale (for linear models)
- **Tree models:** median impute only (scale not required)


In [ ]:
def make_linear_pipe(clf):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", clf),
    ])

def make_tree_pipe(clf):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", clf),
    ])

models = {
    "logistic": make_linear_pipe(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "random_forest": make_tree_pipe(
        RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=30,
            class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
        )
    ),
    "gradient_boosting": make_tree_pipe(
        GradientBoostingClassifier(
            n_estimators=150, max_depth=4, learning_rate=0.08,
            subsample=0.8, random_state=RANDOM_STATE,
        )
    ),
}
print("Models defined:", list(models.keys()))


## 4. Train and evaluate on the validation set

Fit each baseline pipeline on the training fold and report discrimination and calibration-friendly metrics on validation.


In [ ]:
def evaluate(name, pipe, X_tr, y_tr, X_va, y_va):
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    pred = (proba >= 0.5).astype(int)
    metrics = {
        "model": name,
        "roc_auc": roc_auc_score(y_va, proba),
        "pr_auc": average_precision_score(y_va, proba),
        "accuracy": accuracy_score(y_va, pred),
        "precision": precision_score(y_va, pred, zero_division=0),
        "recall": recall_score(y_va, pred, zero_division=0),
        "f1": f1_score(y_va, pred, zero_division=0),
        "brier": brier_score_loss(y_va, proba),
    }
    return metrics, proba, pred, pipe

results = []
fitted = {}
val_probas = {}

for name, pipe in models.items():
    m, proba, pred, fitted_pipe = evaluate(name, pipe, X_train, y_train, X_val, y_val)
    results.append(m)
    fitted[name] = fitted_pipe
    val_probas[name] = proba
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(classification_report(y_val, pred, target_names=["Fully Paid", "Charged Off"]))
    print(f"ROC-AUC: {m['roc_auc']:.4f}  |  PR-AUC: {m['pr_auc']:.4f}  |  Brier: {m['brier']:.4f}")

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("\n" + "="*60)
print("MODEL COMPARISON (validation, threshold=0.5)")
print("="*60)
print(results_df.to_string(index=False))
best_name = results_df.iloc[0]["model"]
print(f"\n>>> Best by ROC-AUC: {best_name}")


In [ ]:
# ROC + PR curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, proba in val_probas.items():
    fpr, tpr, _ = roc_curve(y_val, proba)
    prec, rec, _ = precision_recall_curve(y_val, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, proba):.3f})")
    axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_val, proba):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve")
axes[0].legend(fontsize=9)

axes[1].axhline(y_val.mean(), color="k", ls="--", lw=0.8, label="baseline prevalence")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall curve")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. Threshold tuning (best model)

Default 0.5 is rarely optimal under class imbalance.
We scan thresholds and report operating points:
- **Max F1**
- **Max recall with precision ≥ 0.40**
- **Max precision with recall ≥ 0.50**


In [ ]:
best_proba = val_probas[best_name]
thresholds = np.linspace(0.05, 0.95, 37)

rows = []
for t in thresholds:
    pred_t = (best_proba >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_val, pred_t, zero_division=0),
        "recall": recall_score(y_val, pred_t, zero_division=0),
        "f1": f1_score(y_val, pred_t, zero_division=0),
        "flag_rate": pred_t.mean(),
    })
thr_df = pd.DataFrame(rows)

# Operating points
idx_f1 = thr_df["f1"].idxmax()
cand_prec = thr_df[thr_df["precision"] >= 0.40]
idx_rec = cand_prec["recall"].idxmax() if len(cand_prec) else idx_f1
cand_rec = thr_df[thr_df["recall"] >= 0.50]
idx_prec = cand_rec["precision"].idxmax() if len(cand_rec) else idx_f1

ops = {
    "max_f1": thr_df.loc[idx_f1],
    "max_recall_at_prec_ge_0.40": thr_df.loc[idx_rec],
    "max_precision_at_recall_ge_0.50": thr_df.loc[idx_prec],
}

print(f"Best model: {best_name}")
print("\nOperating points:")
for k, row in ops.items():
    print(f"  {k}: threshold={row['threshold']:.2f}  "
          f"P={row['precision']:.3f}  R={row['recall']:.3f}  "
          f"F1={row['f1']:.3f}  flag_rate={row['flag_rate']:.3f}")

chosen_threshold = float(ops["max_f1"]["threshold"])
print(f"\n>>> Default chosen threshold (max F1): {chosen_threshold:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
ax.plot(thr_df["threshold"], thr_df["recall"], label="Recall")
ax.plot(thr_df["threshold"], thr_df["f1"], label="F1", lw=2)
ax.axvline(chosen_threshold, color="red", ls="--", label=f"chosen={chosen_threshold:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold sweep — {best_name}")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix at chosen threshold
pred_chosen = (best_proba >= chosen_threshold).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, thr, title in [
    (axes[0], 0.5, "Threshold = 0.50"),
    (axes[1], chosen_threshold, f"Threshold = {chosen_threshold:.2f} (max F1)"),
]:
    pred_t = (best_proba >= thr).astype(int)
    cm = confusion_matrix(y_val, pred_t)
    ConfusionMatrixDisplay(cm, display_labels=["Paid", "Default"]).plot(ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(classification_report(y_val, pred_chosen, target_names=["Fully Paid", "Charged Off"]))


## 6. Calibration check

Compare predicted probabilities to observed default rates by bin. Poor calibration does not invalidate ranking but limits use of scores as true PDs.


In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 5))
for name, proba in val_probas.items():
    frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of defaults")
ax.set_title("Calibration curves (validation)")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Segment performance

Evaluate the champion model at the chosen threshold within key underwriting segments (term, renter, score band, missing loan amount).


In [ ]:
val_frame = X_val.copy()
val_frame["Default"] = y_val.values
val_frame["proba"] = best_proba
val_frame["pred"] = pred_chosen

segment_specs = []
if "is_long_term" in val_frame.columns:
    segment_specs.append(("is_long_term", "Long Term"))
if "is_renter" in val_frame.columns:
    segment_specs.append(("is_renter", "Renter"))
if "score_band_poor" in val_frame.columns:
    segment_specs.append(("score_band_poor", "Poor score band"))
if "loan_amount_missing" in val_frame.columns:
    segment_specs.append(("loan_amount_missing", "Loan amount missing"))

seg_rows = []
for col, label in segment_specs:
    for val in [0, 1]:
        m = val_frame[col] == val
        if m.sum() < 30:
            continue
        yt, yp, pr = val_frame.loc[m, "Default"], val_frame.loc[m, "pred"], val_frame.loc[m, "proba"]
        seg_rows.append({
            "segment": label,
            "value": val,
            "n": int(m.sum()),
            "default_rate": yt.mean(),
            "roc_auc": roc_auc_score(yt, pr) if yt.nunique() > 1 else np.nan,
            "recall": recall_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
        })

seg_df = pd.DataFrame(seg_rows)
print(seg_df.to_string(index=False))


## 8. Cross-validated ROC-AUC

Estimate ranking stability with stratified k-fold CV on the full labelled matrix.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in models.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_rows.append({
        "model": name,
        "cv_auc_mean": scores.mean(),
        "cv_auc_std": scores.std(),
        "folds": " ".join(f"{s:.3f}" for s in scores),
    })
    print(f"{name:20s}  CV AUC = {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_rows).sort_values("cv_auc_mean", ascending=False)
print()
print(cv_df.to_string(index=False))


## 9. Refit best model on full data & score test set

Apply the **same cleaning + feature engineering** used in training notebooks,
then restrict to the Standard feature list.


In [ ]:
# --- Reusable feature engineering (aligned with 02_credit_feature_engineering) ---
JOB_MAP = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10, "n/a": np.nan,
}
PURPOSE_GROUPS = {
    "debt_consolidation": "debt_consolidation", "home_improvements": "home",
    "buy_house": "home", "buy_a_car": "vehicle", "business_loan": "business",
    "small_business": "business", "medical_bills": "medical",
    "take_a_trip": "discretionary", "vacation": "discretionary",
    "wedding": "discretionary", "major_purchase": "discretionary",
    "moving": "discretionary", "educational_expenses": "education",
    "renewable_energy": "other", "other": "other",
}

def engineer_raw(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.dropna(how="all").copy()
    if "Loan ID" in d.columns:
        d = d.drop_duplicates(subset=["Loan ID"])
    if "Credit Score" in d.columns:
        mask = d["Credit Score"] > 850
        d.loc[mask, "Credit Score"] = d.loc[mask, "Credit Score"] / 10
    if "Current Loan Amount" in d.columns:
        d["Current Loan Amount"] = d["Current Loan Amount"].replace(99999999, np.nan)
    if "Home Ownership" in d.columns:
        d["Home Ownership"] = d["Home Ownership"].replace({"HaveMortgage": "Home Mortgage"})
    if "Purpose" in d.columns:
        d["Purpose"] = d["Purpose"].str.lower().str.strip().str.replace(" ", "_")

    d["job_years"] = d["Years in current job"].map(JOB_MAP) if "Years in current job" in d.columns else np.nan
    d["job_years_missing"] = d["job_years"].isna().astype(int)
    d["is_short_tenure"] = (d["job_years"].fillna(99) <= 1).astype(int)

    for col, alias in [
        ("Credit Score", "credit_score"), ("Annual Income", "annual_income"),
        ("Current Loan Amount", "loan_amount"),
        ("Months since last delinquent", "months_delinquent"),
        ("Bankruptcies", "bankruptcies"), ("Maximum Open Credit", "max_open_credit"),
    ]:
        if col in d.columns:
            d[f"{alias}_missing"] = d[col].isna().astype(int)

    d["dti"] = np.where(d.get("Annual Income", 0) > 0,
                        (d["Monthly Debt"] * 12) / d["Annual Income"], np.nan) if "Monthly Debt" in d.columns else np.nan
    d["lti"] = np.where(d.get("Annual Income", 0) > 0,
                        d["Current Loan Amount"] / d["Annual Income"], np.nan)
    d["credit_util"] = np.where(
        d.get("Maximum Open Credit", 0) > 0,
        d["Current Credit Balance"] / d["Maximum Open Credit"], np.nan
    ) if "Current Credit Balance" in d.columns else np.nan
    if "credit_util" in d.columns:
        d["credit_util"] = d["credit_util"].clip(upper=2.0)
    d["debt_to_loan"] = np.where(
        d.get("Current Loan Amount", 0) > 0,
        d["Monthly Debt"] / d["Current Loan Amount"], np.nan
    ) if "Monthly Debt" in d.columns else np.nan
    d["balance_to_income"] = np.where(
        d.get("Annual Income", 0) > 0,
        d["Current Credit Balance"] / d["Annual Income"], np.nan
    ) if "Current Credit Balance" in d.columns else np.nan

    d["has_delinquency_hist"] = d["Months since last delinquent"].notna().astype(int) if "Months since last delinquent" in d.columns else 0
    d["has_credit_problems"] = (d["Number of Credit Problems"].fillna(0) > 0).astype(int) if "Number of Credit Problems" in d.columns else 0
    d["has_bankruptcy"] = (d["Bankruptcies"].fillna(0) > 0).astype(int) if "Bankruptcies" in d.columns else 0
    d["has_tax_lien"] = (d["Tax Liens"].fillna(0) > 0).astype(int) if "Tax Liens" in d.columns else 0
    d["has_any_public_record"] = ((d["has_credit_problems"] + d["has_bankruptcy"] + d["has_tax_lien"]) > 0).astype(int)

    d["is_long_term"] = (d["Term"] == "Long Term").astype(int) if "Term" in d.columns else 0
    d["is_renter"] = (d["Home Ownership"] == "Rent").astype(int) if "Home Ownership" in d.columns else 0
    d["is_mortgage"] = (d["Home Ownership"] == "Home Mortgage").astype(int) if "Home Ownership" in d.columns else 0
    d["is_homeowner"] = (d["Home Ownership"] == "Own Home").astype(int) if "Home Ownership" in d.columns else 0

    if "Purpose" in d.columns:
        d["purpose_group"] = d["Purpose"].map(PURPOSE_GROUPS).fillna("other")
        for grp in ["debt_consolidation", "home", "business", "discretionary", "medical", "vehicle", "other"]:
            d[f"purpose_{grp}"] = (d["purpose_group"] == grp).astype(int)

    def qbin(s, q=5):
        try:
            return pd.qcut(s, q=q, duplicates="drop").cat.codes.replace(-1, np.nan)
        except Exception:
            return pd.Series(np.nan, index=s.index)

    d["credit_score_bin"] = qbin(d["Credit Score"]) if "Credit Score" in d.columns else np.nan
    d["income_bin"] = qbin(d["Annual Income"]) if "Annual Income" in d.columns else np.nan
    d["loan_amount_bin"] = qbin(d["Current Loan Amount"]) if "Current Loan Amount" in d.columns else np.nan
    d["dti_bin"] = qbin(d["dti"]) if "dti" in d.columns else np.nan
    d["util_bin"] = qbin(d["credit_util"]) if "credit_util" in d.columns else np.nan

    def score_band(s):
        if pd.isna(s):
            return "missing"
        if s < 650:
            return "poor"
        if s < 700:
            return "fair"
        if s < 740:
            return "good"
        return "excellent"

    if "Credit Score" in d.columns:
        d["credit_score_band"] = d["Credit Score"].apply(score_band)
        for band in ["poor", "fair", "good", "excellent", "missing"]:
            d[f"score_band_{band}"] = (d["credit_score_band"] == band).astype(int)

    d["longterm_x_renter"] = d["is_long_term"] * d["is_renter"]
    d["longterm_x_lowscore"] = d["is_long_term"] * d.get("score_band_poor", 0)
    med_dti = d["dti"].median() if "dti" in d.columns else np.nan
    d["high_dti_x_renter"] = ((d["dti"] > med_dti) & (d["is_renter"] == 1)).astype(int) if "dti" in d.columns else 0
    d["short_tenure_x_longterm"] = d["is_short_tenure"] * d["is_long_term"]

    for col, alias in [
        ("Annual Income", "log_annual_income"), ("Current Loan Amount", "log_loan_amount"),
        ("Monthly Debt", "log_monthly_debt"), ("Current Credit Balance", "log_credit_balance"),
        ("Maximum Open Credit", "log_max_open_credit"),
    ]:
        if col in d.columns:
            d[alias] = np.log1p(d[col].clip(lower=0))

    return d

print("Feature engineering function ready.")


In [ ]:
# Refit best model on full training matrix
best_pipe = models[best_name]
best_pipe.fit(X, y)
print(f"Refitted {best_name} on full train ({len(X):,} rows)")

# Score test
test_raw = pd.read_csv(RAW_TEST)
test_eng = engineer_raw(test_raw)
print(f"Test engineered: {test_eng.shape}")

# Align columns
for c in feature_cols:
    if c not in test_eng.columns:
        test_eng[c] = np.nan
X_test = test_eng[feature_cols].copy()

test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= chosen_threshold).astype(int)

test_ids = test_eng[["Loan ID", "Customer ID"]].copy() if "Loan ID" in test_eng.columns else pd.DataFrame(index=X_test.index)
submission = test_ids.copy()
submission["Default_Probability"] = np.round(test_proba, 4)
submission["Predicted_Default"] = test_pred

print(f"Test predictions: {len(submission):,}")
print(f"Predicted default rate @ thr={chosen_threshold:.2f}: {test_pred.mean()*100:.2f}%")
print(submission.head())


## 10. Persist artifacts

Write comparison tables, threshold sweep, test predictions, and the fitted model pickle for downstream scoring.


In [ ]:
results_df.to_csv(OUT_DIR / "model_comparison_val.csv", index=False)
cv_df.to_csv(OUT_DIR / "model_cv_auc.csv", index=False)
thr_df.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)
submission.to_csv(OUT_DIR / "test_predictions.csv", index=False)

with open(OUT_DIR / f"best_model_{best_name}.pkl", "wb") as f:
    pickle.dump(best_pipe, f)

meta = {
    "best_model": best_name,
    "threshold": chosen_threshold,
    "features": feature_cols,
    "val_metrics": results_df[results_df["model"] == best_name].iloc[0].to_dict(),
    "cv_auc": cv_df[cv_df["model"] == best_name].iloc[0].to_dict(),
    "operating_points": {k: v.to_dict() for k, v in ops.items()},
}
with open(OUT_DIR / "model_meta.json", "w") as f:
    json.dump(meta, f, indent=2, default=float)

print("Saved to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.iterdir()):
    print(f"  - {p.name}")


## 11. Summary

### Results at a glance

- Models compared on the **Standard (25-feature)** set with a stratified 80/20 split
- Ranking metric: **ROC-AUC**; threshold chosen by **max F1** on validation
- Best model refit on full training data and applied to `credit_test.csv`

### Next steps (optional)

1. Hyperparameter search (GridSearchCV / RandomizedSearchCV) on the best family
2. Probability calibration (`CalibratedClassifierCV`) if Brier score is high
3. Cost-sensitive threshold using explicit false-positive / false-negative costs
4. Model card + monitoring plan (PSI on key features, monthly AUC drift)


In [ ]:
print("=" * 60)
print("MODELLING COMPLETE")
print("=" * 60)
print(f"Best model:     {best_name}")
print(f"Val ROC-AUC:    {results_df.iloc[0]['roc_auc']:.4f}")
print(f"Val PR-AUC:     {results_df.iloc[0]['pr_auc']:.4f}")
print(f"Threshold:      {chosen_threshold:.2f}")
print(f"CV ROC-AUC:     {cv_df[cv_df['model']==best_name]['cv_auc_mean'].values[0]:.4f} "
      f"± {cv_df[cv_df['model']==best_name]['cv_auc_std'].values[0]:.4f}")
print(f"Test scored:    {len(submission):,}")
print(f"Artifacts:      {OUT_DIR.resolve()}")
print("=" * 60)
